In [23]:
import time
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

# ========== 1. Configurar Chrome (ventana visible para pasar anti-bot) ==========
chrome_options = Options()
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

# ========== 2. Cargar la página ==========
driver.get("https://listado.mercadolibre.com.co/laptops")
print("Esperando carga...")
time.sleep(10)

soup = BeautifulSoup(driver.page_source, 'html.parser')
driver.quit()

# ========== 3. Seleccionar TODOS los productos ==========
productos = soup.find_all('li', class_='ui-search-layout__item')
print(f"Productos encontrados: {len(productos)}")

# ========== 4. Extraer campos ==========
datos = []

for prod in productos:  # ← SIN [:20], ahora extrae todos
    # Título
    titulo_tag = prod.find('a', class_='poly-component__title')
    if not titulo_tag:
        titulo_tag = prod.find('h3', class_='poly-component__title-wrapper')
    titulo = titulo_tag.text.strip() if titulo_tag else "No disponible"
    
    # Precio (limpiar formato: quitar puntos y convertir a número)
    precio_texto = "No disponible"
    precio_num = None
    precio_tag = prod.find('span', class_='andes-money-amount__fraction')
    if precio_tag:
        precio_texto = f"$ {precio_tag.text.strip()}"
        try:
            precio_num = int(precio_tag.text.strip().replace('.', ''))
        except ValueError:
            precio_num = None
    
    # Vendedor
    vendedor_tag = prod.find('span', class_='poly-component__seller')
    vendedor = vendedor_tag.text.strip() if vendedor_tag else "No especificado"
    
    # Descuento
    descuento_tag = prod.find('span', class_='poly-price__discount-polylabel')
    descuento = descuento_tag.text.strip() if descuento_tag else "Sin descuento"
    
    # Enlace limpio (sin parámetros de tracking)
    enlace = "No disponible"
    enlace_tag = prod.find('a', class_='poly-component__title')
    if enlace_tag and 'href' in enlace_tag.attrs:
        enlace = enlace_tag['href'].split('#')[0]
    
    datos.append({
        'Producto': titulo,
        'Precio': precio_texto,
        'Precio_Numerico': precio_num,  # ← útil para análisis
        'Vendedor': vendedor,
        'Descuento': descuento,
        'Enlace': enlace
    })

# ========== 5. Crear DataFrame ==========
df = pd.DataFrame(datos)
print(f"\nSe extrajeron {len(df)} registros\n")

# ========== 6. Correcciones estéticas ==========
# Truncar títulos largos solo para VISUALIZACIÓN (el CSV guarda completo)
df_visual = df.copy()
df_visual['Producto'] = df_visual['Producto'].apply(
    lambda x: x[:55] + '...' if len(x) > 55 else x
)

# Mostrar solo las columnas principales para visualización
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
print(df_visual[['Producto', 'Precio', 'Vendedor', 'Descuento']].to_string(index=True))

# ========== 7. Guardar CSV COMPLETO (sin truncar) ==========
ruta_csv = os.path.join(os.path.expanduser("~"), "Desktop", "laptops_mercadolibre.csv")
df.to_csv(ruta_csv, index=False, encoding='utf-8-sig')
print(f"\n CSV guardado en: {ruta_csv}")
print(f"   Total de registros: {len(df)}")
print(f"   Columnas: {list(df.columns)}")

Esperando carga...
Productos encontrados: 60

Se extrajeron 60 registros

                                                      Producto       Precio         Vendedor      Descuento
0   Portátil HP Omnibook 3 IA 14-ht0031la Intel C3 8gb 512g...  $ 4.099.000    Mercado Libre        43% OFF
1   Portátil HP Omnibook 14-hu0050la Intel Core5 120U 8gb 5...  $ 3.499.000    Mercado Libre        44% OFF
2   Portátil HP OmniBook 3 14-hy0030la AMD Ryzen 3 8 GB RAM...  $ 3.199.000    Mercado Libre        40% OFF
3   Portátil HP IA Intel Core Ultra 5 225U 8GB 512GB. 15.6’...  $ 3.499.000    Mercado Libre        37% OFF
4   Lenovo Ideapad Slim 3 15irh10 I5-13420h 16gb 512gb Tact...  $ 3.299.999  No especificado        35% OFF
5   Portátil Hp 255 G10 Ryzen 3 7320u 16gb 512gb Hd 15.6 Ne...  $ 1.850.000  No especificado        19% OFF
6                Apple Macbook Neo 13" - (256gb) - Azul Índigo  $ 3.599.900    Mercado Libre        27% OFF
7   Notebook Asus Vivobook S16 Ryzen 9 16gb Ram 512 Gb Ssd ...

In [24]:
import requests
from bs4 import BeautifulSoup

url = "https://listado.mercadolibre.com.co/laptops"
headers = {"User-Agent": "Mozilla/5.0 ..."}
r = requests.get(url, headers=headers)
print(len(r.text))  
print('ui-search-layout' in r.text) 

22392
False


In [ ]:
import subprocess
import os
import pandas as pd

# Ruta del proyecto
proyecto = r"C:\Users\Edwin Ruiz\libros_scraper"

# Ejecutar Scrapy
resultado = subprocess.run(
    ["scrapy", "crawl", "laptops"],
    cwd=proyecto,
    capture_output=True,
    text=True,
    timeout=300
)

print("=== STDOUT ===")
print(resultado.stdout)
print("\n=== STDERR ===")
print(resultado.stderr)

# Leer el CSV generado
ruta_csv = os.path.join(proyecto, "laptops_scrapy.csv")
if os.path.exists(ruta_csv):
    df_scrapy = pd.read_csv(ruta_csv)
    print(f"\n✅ Registros extraídos con Scrapy: {len(df_scrapy)}")
    print(df_scrapy.head(10).to_string())
else:
    print("\n❌ El CSV no se generó. Scrapy fue bloqueado por Mercado Libre.")
    print("   Esto es evidencia del TLS fingerprinting del sitio.")